In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.signal import butter, filtfilt
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold     

file_path = r"D:\ML project\guided\guided_dataset_X.npy"
data = np.load(file_path)

#We start by doing a filtering of our data. "https://www.sciencedirect.com/science/article/pii/S0021929010000631" this paper is a relevant litterature on the subject
#We learn that most of the noise that occurs when recording our sEMG is located in lower frequencies. And that a high-pass filtering at 10hz is highly beneficial with diminishing performance at 20 and 30 hz 
#while still being beneficial as not much revelant signal information is lost.
#As the choice of the frequencie of the high-pass is muscle dependent and as we are limited by our lack of expertise. We will follow the recommendation of the paper and settle at doing a Butterworth filter with a corner frequency of 20 Hz and a slope of 12 dB/oct 
#To do so we will use the butter and filtfilt functions of the scipy library

#In addition, we also learn from this paper "https://iopscience.iop.org/article/10.1088/1742-6596/1237/3/032008/pdf" that most of the sEMG's energy is concentrated in the 0-500hz range 
#Our sampling frenquencie being of 1024hz, twice the bandwith of the signal, by the Nyquist-Shannon theorem we can efficiently control for aliasing up to the 500hz frequencies.
#Even if according to the first paper most of the noise is located in lower frequencies, since relveant information is under the 500hz threshold, ça mange pas de pain to control for higher frequencies.


cutoff_high = 20       # We select (in hz) the cutoff of our high-pass filter
cutoff_low = 500     # We select (in hz) the cutoff of our low-pass filter
order_high = 2         # We select the number of poles. In the Butterworth filter, each pole correspong to 6db, so we select two to get the recommended 12dp/octave
order_low = 4        # Once again we select the number of poles. In the paper the nbr of decibel/octave for the high-pass filter was 24 dB/oct, so we follow this recommendation.

b_high, a_high = butter(order_high, cutoff_high / (1024 / 2), btype='highpass') #The butter function returns the filter coefficients a (denominator),b (numerator). And take as inupts the order and Wn, the desired cutoff frequency being the frequency at which at which the magnitude response of the filter is 1 / √2
b_low, a_low = butter(order_low, cutoff_low / (1024 / 2), btype='lowpass')

first_filtering = filtfilt(b_high, a_high, data)  #Here we use the filtfilt function, which controls for distortion. That filters forward the backward the forward then blabla, à compléter, pas encore parfaitement compris. https://dsp.stackexchange.com/questions/9467/what-is-the-advantage-of-matlabs-filtfilt 
final_filtering = filtfilt(b_low, a_low, first_filtering)

n_sessions, n_electrodes, n_samples = data.shape

for session in range(n_sessions):
    for electrode in range(n_electrodes):
        signal    = final_filtering[session, electrode, :]
        signal_o  = data[session, electrode, :]




#Dois encore copier coller les sources.
#Peut être modifier le dernier loop


In [ ]:
#We start by "windowing" our data set by creating overlapping windows of size 500ms with a 250ms step betwwen each windows
#this results in the creation of 919 session per electrode, 6433 per session and a total of 25732

ws = 500                # Size of our windows
step = 250              # Distance between the beginning of each windows 2 by 2
n_samples = 230000      # Total time of a session

#To do so we use a loop in a loop that will go over the data of each electrodes of each session and add it in a two level dictionnary
windowsEMG = {
    s: {                                                                      
        e: [data[s, e, i:i+ws] for i in range(0, n_samples - ws + 1, step)]    
        for e in range(7)
    }
    for s in range(5)
}


In [ ]:
#In our situation a K-fold cross validation would not be adapted since it would randomly splits the data.
#But in our case, data is structured and we use overlapping windows which would cause a high risk of windows from the same session to end up in training and validation.
#As in our situation, our data is organized in session, we will use a Leave-One-Session-Out cross-validation method

windows_list = []
session_index = []
for s in range(5):
    for e in range(7):
        for window in windowsEMG[s][e]:  #list of windows for electrode e of session s
            windows_list.append(window)  #we add all those windows to a list 
            session_index.append(s)      #and we attribute to each an index depending on the wession it's from 

windows_list = np.array(windows_list)    #we convert the list we just create in a NumPy array so that we can use the GroupKFold function
groups = np.array(session_index)         #same for the indexes

group_kfold = GroupKFold(n_splits=5)     #we use the GroupKFold function to create our folds, every iteration with one entire session in each
